In [36]:
import asyncio
import json
import os
import shutil
from collections.abc import Sequence
from datetime import UTC
from datetime import datetime
from decimal import Decimal
from getpass import getpass
from hashlib import sha256
from pathlib import Path
from tempfile import mkdtemp

import pandas as pd

from nautilus_trader.adapters.finfeed import FinFeedDataLoader
from nautilus_trader.backtest import BacktestNode
from nautilus_trader.common import LogLevel
from nautilus_trader.config import BacktestDataConfig
from nautilus_trader.config import BacktestEngineConfig
from nautilus_trader.config import BacktestRunConfig
from nautilus_trader.config import BacktestVenueConfig
from nautilus_trader.config import DataCatalogConfig
from nautilus_trader.config import DataEngineConfig
from nautilus_trader.config import LoggerConfig
from nautilus_trader.execution import MakerTakerFeeModel
from nautilus_trader.model import AccountType
from nautilus_trader.model import BinaryOption
from nautilus_trader.model import BookAction
from nautilus_trader.model import BookType
from nautilus_trader.model import InstrumentId
from nautilus_trader.model import NautilusDataType
from nautilus_trader.model import OmsType
from nautilus_trader.model import OrderBook
from nautilus_trader.model import OrderBookDelta
from nautilus_trader.model import Quantity
from nautilus_trader.model import RecordFlag
from nautilus_trader.model import Venue
from nautilus_trader.persistence import ParquetDataCatalog
from nautilus_trader.trading import OrderBookImbalanceConfig

In [47]:
MARKET_ID = "BITCOIN-ABOVE-76K-ON-NOVEMBER-3-2026_YES"
INSTRUMENT_ID = InstrumentId.from_str(f"{MARKET_ID}.FINFEED_POLYMARKET")
START = datetime(2026, 11, 2, 20, 0, 5, tzinfo=UTC)
END = datetime(2026, 11, 2, 20, 2, tzinfo=UTC)
# DATA_DIR = Path(os.environ.get("NAUTILUS_DATA_DIR", "~/Downloads/Data")).expanduser() / "FinFeed"
DATA_DIR = Path.cwd().resolve() / "Downloads/Data/FinFeed"
catalog_path = DATA_DIR / "btc-above-76k-nov3-20261102-20"
book_type = BookType.L2_MBP

In [48]:
from dotenv import load_dotenv
load_dotenv()

True

# orderbook data

In [49]:
async def collect_book(api_key: str) -> tuple[BinaryOption, list[OrderBookDelta]]:
    """
    Download the outcome definition and its historical book deltas.
    """
    loader = FinFeedDataLoader(api_key=api_key)

    instrument = await loader.request_instrument(INSTRUMENT_ID)
    deltas = await loader.request_book_deltas(
        INSTRUMENT_ID,
        start=pd.Timestamp(START).value,
        end=pd.Timestamp(END).value,
    )
    return instrument, deltas


def book_digest(deltas: Sequence[OrderBookDelta], instrument: BinaryOption) -> str:
    """
    Hash delta values with the instrument's precision, including zero CLEAR fields.
    """
    digest = sha256()

    for delta in deltas:
        record = delta.to_dict()
        record["order"]["price"] = format(
            delta.order.price.as_decimal(),
            f".{instrument.price_precision}f",
        )
        record["order"]["size"] = format(
            delta.order.size.as_decimal(),
            f".{instrument.size_precision}f",
        )
        digest.update(json.dumps(record, sort_keys=True).encode())
    return digest.hexdigest()


In [ ]:
print(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
selection = {"market_id": MARKET_ID, "start": START.isoformat(), "end": END.isoformat()}

if not catalog_path.exists():
    api_key = os.environ["FINFEED_API_KEY"] or getpass("FinFeed Prediction Markets API key: ")
    instrument, deltas =  await collect_book(api_key)
    
    # write to a temp file first, so a failed leaves no reusable cache
    temp_path = Path(mkdtemp(prefix="finfeed-catalog-", dir=DATA_DIR))
    print('temp_path:',temp_path)
    try:
        catalog = ParquetDataCatalog(str(temp_path))
        catalog.write_instruments([instrument])
        catalog.write_order_book_deltas(deltas)

        manifest = {
            **selection,
            "deltas": len(deltas),
            "sha256": book_digest(deltas, instrument),
        }

        (temp_path / "manifest.json").write_text(
            json.dumps(manifest, indent=2),
            encoding="utf-8",
        )
        del catalog
        temp_path.rename(catalog_path) # move temp to final
    finally:
        if temp_path.exists():
            shutil.rmtree(temp_path)

catalog = ParquetDataCatalog(str(catalog_path)) 
manifest = json.loads((catalog_path / "manifest.json").read_text(encoding="utf-8"))
assert {key: manifest[key] for key in selection} == selection
[instrument] = catalog.instruments(instrument_ids=[str(INSTRUMENT_ID)])
deltas = catalog.query_order_book_deltas(identifiers=[str(INSTRUMENT_ID)])

D:\horiz\IMPORTANT\0Quant\project_PM\sample_codes\nautilus\Downloads\Data\FinFeed


In [ ]:
assert instrument.id == INSTRUMENT_ID
assert instrument.outcome == "Yes"
assert instrument.expiration_ns > pd.Timestamp(END).value
assert len(deltas) == manifest["deltas"]
assert deltas[0].action == BookAction.CLEAR
assert RecordFlag.F_SNAPSHOT.matches(deltas[0].flags)
assert RecordFlag.F_LAST.matches(deltas[-1].flags)
assert book_digest(deltas, instrument) == manifest["sha256"]

In [ ]:
# rebuild order book

book = OrderBook(INSTRUMENT_ID, book_type)
top_of_book_rows = []
batch_count = 0

for delta in deltas:
    book.apply_delta(delta)
    if not RecordFlag.F_LAST.matches(delta.flags):
        continue
    book.check_integrity()
    batch_count += 1
    bid = book.best_bid_price()
    ask = book.best_ask_price()
    bid_size = book.best_bid_size()
    ask_size = book.best_ask_size()
    if bid is None or ask is None or bid_size is None or ask_size is None:
        continue

    if ask <= bid:
        continue
    bid_qty = bid_size.as_decimal()
    ask_qty = ask_size.as_decimal()
    top_of_book_rows.append(
        {
            "timestamp": delta.ts_init,
            "bid": bid.as_decimal(),
            "ask": ask.as_decimal(),
            "spread": ask.as_decimal() - bid.as_decimal(),
            "bid_size": bid_qty,
            "ask_size": ask_qty,
            "imbalance": (bid_qty - ask_qty) / (bid_qty + ask_qty),
        },
    )
# Sanity checks: several batches, usable rows, valid final book
assert batch_count > 1
assert top_of_book_rows
assert book.best_bid_price() is not None
assert book.best_ask_price() is not None
assert book.spread() > 0



Catalog: D:\horiz\IMPORTANT\0Quant\project_PM\sample_codes\nautilus\Downloads\Data\FinFeed\btc-above-76k-nov3-20261102-20
Instrument: BITCOIN-ABOVE-76K-ON-NOVEMBER-3-2026_YES.FINFEED_POLYMARKET, outcome: Yes
Loaded 13,900 deltas in 13,774 completed batches
Delta SHA-256: df25d4f73bb70f3543407902776e4acaeb26cb630cef8d6bf1b21e0060322230
                                        bid     ask  spread  bid_size   ask_size                        imbalance
timestamp                                                                                                        
2026-11-02 20:00:05.000000000+00:00  0.7510  0.7570  0.0060  9.000000  15.140000  -0.2543496271748135874067937034
2026-11-02 20:00:05.851031800+00:00  0.7510  0.7570  0.0060  9.000000  15.140000  -0.2543496271748135874067937034
2026-11-02 20:00:05.851181800+00:00  0.7510  0.7570  0.0060  9.000000  15.140000  -0.2543496271748135874067937034
2026-11-02 20:00:05.851488900+00:00  0.7510  0.7570  0.0060  9.000000  15.140000  -0.25434962

In [55]:
# output & summary

top_of_book = pd.DataFrame(top_of_book_rows)
top_of_book.index = pd.to_datetime(top_of_book.pop("timestamp"), utc=True)
print(f"Catalog: {catalog_path}")
print(f"Instrument: {instrument.id}, outcome: {instrument.outcome}")
print(f"Loaded {len(deltas):,} deltas in {batch_count:,} completed batches")
print(f"Delta SHA-256: {manifest['sha256']}")
print(top_of_book.head().to_string())
print(f"Spread range: {top_of_book['spread'].min()} to {top_of_book['spread'].max()} pUSD")

Catalog: D:\horiz\IMPORTANT\0Quant\project_PM\sample_codes\nautilus\Downloads\Data\FinFeed\btc-above-76k-nov3-20261102-20
Instrument: BITCOIN-ABOVE-76K-ON-NOVEMBER-3-2026_YES.FINFEED_POLYMARKET, outcome: Yes
Loaded 13,900 deltas in 13,774 completed batches
Delta SHA-256: df25d4f73bb70f3543407902776e4acaeb26cb630cef8d6bf1b21e0060322230
                                        bid     ask  spread  bid_size   ask_size                        imbalance
timestamp                                                                                                        
2026-11-02 20:00:05.000000000+00:00  0.7510  0.7570  0.0060  9.000000  15.140000  -0.2543496271748135874067937034
2026-11-02 20:00:05.851031800+00:00  0.7510  0.7570  0.0060  9.000000  15.140000  -0.2543496271748135874067937034
2026-11-02 20:00:05.851181800+00:00  0.7510  0.7570  0.0060  9.000000  15.140000  -0.2543496271748135874067937034
2026-11-02 20:00:05.851488900+00:00  0.7510  0.7570  0.0060  9.000000  15.140000  -0.25434962

# backtesting

In [ ]:
config = BacktestRunConfig(
    engine=BacktestEngineConfig(
        logging=LoggerConfig(stdout_level=LogLevel.ERROR),
        data_engine=DataEngineConfig(emit_quotes_from_book=True),
    ),
    data=[
        BacktestDataConfig(
            catalog=DataCatalogConfig(path=str(catalog_path)),
            data_type=NautilusDataType.OrderBookDelta,
            instrument_id=instrument.id,
        ),
    ],
    venues=[
        BacktestVenueConfig(
            name="FINFEED_POLYMARKET",
            oms_type=OmsType.NETTING,
            account_type=AccountType.CASH,
            base_currency=instrument.currency,
            starting_balances=["1000 pUSD"],
            book_type=book_type,
            fee_model=MakerTakerFeeModel(maker_rate=Decimal(0), taker_rate=Decimal(0)),
        ),
    ],
    dispose_on_completion=False,
)

# built-in Rust orderbookImbalance strategy
strategy_config = OrderBookImbalanceConfig(
    instrument_id=instrument.id,
    book_type=book_type,
    max_trade_size=Quantity.from_str("1.000000"),
    trigger_min_size=Decimal("100.0"),
    trigger_imbalance_ratio=Decimal("0.20"),
    min_seconds_between_triggers=1.0,
)

In [56]:
node = BacktestNode(configs=[config])
try:
    node.build()
    node.add_builtin_strategy(config.id, "OrderBookImbalance", strategy_config)
    results = node.run()
    orders = node.generate_orders_report(config.id)
    executions = node.generate_fills_report(config.id)
    positions = node.generate_positions_report(config.id)
    account = node.generate_account_report(config.id, venue=Venue("FINFEED_POLYMARKET"))
    print(results)
    print(orders["status"].value_counts().to_string())
    print(executions[["order_side", "order_type", "last_qty", "last_px", "commission"]].head())
    print(
        positions[
            [
                "side",
                "quantity",
                "peak_qty",
                "avg_px_open",
                "avg_px_close",
                "realized_pnl",
            ]
        ].to_string(),
    )
    print(account[["total", "locked", "free", "currency"]].tail().to_string())
    assert not positions.empty, "No positions opened; inspect the selected window and signal"
    assert positions["ts_closed"].notna().all(), "The strategy left an open position"
    assert positions["side"].eq("FLAT").all()
    assert positions["quantity"].map(Decimal).eq(Decimal(0)).all()
finally:
    node.dispose()

[BacktestResult(trader_id='TRADER-001', elapsed=114.91s, iterations=13774, orders=37, positions=1)]
status
REJECTED    35
FILLED       2
                             order_side order_type  last_qty last_px  \
client_order_id                                                        
O-20261102-200058-001-000-13        BUY      LIMIT  1.000000  0.7620   
O-20261102-200102-001-000-14       SELL      LIMIT  1.000000  0.7440   

                                 commission  
client_order_id                              
O-20261102-200058-001-000-13  0.000000 pUSD  
O-20261102-200102-001-000-14  0.000000 pUSD  
                                                                                    side  quantity  peak_qty avg_px_open avg_px_close    realized_pnl
position_id                                                                                                                                          
BITCOIN-ABOVE-76K-ON-NOVEMBER-3-2026_YES.FINFEED_POLYMARKET-OrderBookImbalance-000  FLAT  

# My own strategy

In [ ]:
from strategy.maker_rebate import PolyMakerConfig, PolyMakerStrategy
# strategy from warproxx
strategy = PolyMakerStrategy(PolyMakerConfig(
    instrument_id=instrument.id,
    book_type=book_type,
    reduce_only_hours=1.0,
    halt_before_hours=0.5,
))

node = BacktestNode(configs=[config])
try:
    node.build()
    node.add_strategy(config.id, strategy)
    print(node.run())
    print(dict(strategy.regime_counts))
    print(node.generate_orders_report(config.id)[["side", "quantity", "price", "status"]])
finally:
    node.dispose()

[BacktestResult(trader_id='TRADER-001', elapsed=114.91s, iterations=13774, orders=11, positions=0)]
{'quiet': 135, 'event': 176}
                             side   quantity   price    status
client_order_id                                               
O-20261102-200005-001-001-1   BUY  29.290000  0.7510  CANCELED
O-20261102-200006-001-001-2   BUY  29.450000  0.7470  CANCELED
O-20261102-200009-001-001-3   BUY  29.290000  0.7510  CANCELED
O-20261102-200010-001-001-4   BUY  29.450000  0.7470  CANCELED
O-20261102-200013-001-001-5   BUY  29.650000  0.7420  CANCELED
O-20261102-200013-001-001-6   BUY  29.690000  0.7410  CANCELED
O-20261102-200021-001-001-7   BUY  29.890000  0.7360  CANCELED
O-20261102-200131-001-001-8   BUY  29.530000  0.7450  CANCELED
O-20261102-200133-001-001-9   BUY  29.690000  0.7410  CANCELED
O-20261102-200151-001-001-10  BUY  29.850000  0.7370  CANCELED
O-20261102-200157-001-001-11  BUY  30.100000  0.7310  CANCELED
